In [1]:
import itertools
import json
import os

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, 
    accuracy_score, 
    roc_auc_score, 
    roc_curve,
)

from tqdm.notebook import (
    tqdm, 
    trange,
)

# labeled data

In [2]:
IS_RESAMPLE = False
main_path = os.getcwd()
labeling_path = os.path.join(
    main_path,
    'project-1-at-2025-05-13-11-10-34463d27.json'
)
audio_sets_path = os.path.join(
    os.path.dirname(os.getcwd()),
    'Phonetics Project (Praat)'
    )
for _path in (main_path, labeling_path, audio_sets_path):
    display(_path)

'/home/alexeysvatov/projects/ProjectLinguist/jupyter/scripts'

'/home/alexeysvatov/projects/ProjectLinguist/jupyter/scripts/project-1-at-2025-05-13-11-10-34463d27.json'

'/home/alexeysvatov/projects/ProjectLinguist/jupyter/Phonetics Project (Praat)'

In [3]:
def preprocess_audio(audio_file_path, hop_length=256, n_mels=20):
    audio, sr = librosa.load(audio_file_path)
    _preprocessed = {
        'melspectrum': np.mean(librosa.feature.melspectrogram(
            y=audio,
            sr=sr,
            hop_length=hop_length, 
            n_mels=n_mels
        ), axis=1),
        'mfcc': np.mean(librosa.feature.mfcc(
            y=audio, 
            sr=sr,
            hop_length=hop_length,
            n_mels=n_mels
        ), axis=1),
        'rolloff': abs(np.mean(librosa.feature.spectral_rolloff(
            y=audio, 
            sr=sr
        ))),
        'zrate': abs(np.mean(librosa.feature.zero_crossing_rate(audio))),
    }
    
    return _preprocessed

In [4]:
labels_df = pd.read_json(labeling_path)
labels_df = labels_df.loc[:, ['file_upload', 'annotations']].explode('annotations', ignore_index=True)
labels_df['file_upload'] = labels_df['file_upload'].apply(lambda _: _.partition('-')[2])
labels_df['result'] = labels_df['annotations'].apply(lambda _: _['result'] if isinstance(_, dict) else _)
labels_df = labels_df.drop('annotations', axis=1).explode('result', ignore_index=True)
labels_df['start'] = labels_df['result'].apply(lambda _: _['value']['start'] if isinstance(_, dict) else _)
labels_df['end'] = labels_df['result'].apply(lambda _: _['value']['end'] if isinstance(_, dict) else _)
labels_df['difference'] = (labels_df['end'] - labels_df['start']) * 1000
labels_df['label'] = labels_df['result'].apply(lambda _: _['value']['labels'] if isinstance(_, dict) else _)
labels_df = labels_df.drop('result', axis=1).explode('label', ignore_index=True)
labels_df = labels_df.loc[(labels_df.label != 'ignore') & (labels_df.label != 'noise') & (labels_df.end - labels_df.start >= 0.1), :]
labels_df['colab_file_path'] = audio_sets_path + '/' + labels_df['file_upload'].str.partition('_')[0] + '_' + labels_df['file_upload'].str.partition('_')[2].str.replace('_', ' ')
labels_df['label'] = pd.Categorical(labels_df['label'])
labels_df['label_id'] = labels_df['label'].cat.codes
labels_df.head()

,file_upload,start,end,difference,label,colab_file_path,label_id
2,1_In_a_restaurant.mp3,5.900917,6.019838,118.921208,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18
3,1_In_a_restaurant.mp3,6.230580,6.454639,224.059099,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18
4,1_In_a_restaurant.mp3,6.019880,6.230599,210.718572,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15
5,1_In_a_restaurant.mp3,6.454629,6.617893,163.263443,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15
6,1_In_a_restaurant.mp3,6.617900,6.870468,252.567449,low_rise,/home/alexeysvatov/projects/ProjectLinguist/ju...,10


In [5]:
# makes labeled slices
labels_df['colab_samples_path'] = None
colab_file_path = None
colab_samples_path = os.path.join(main_path, 'labeled_samples')
for idx in tqdm(labels_df.index):
    audio_slice_file_path = os.path.join(colab_samples_path,
                                         f"{idx}_{labels_df.loc[idx, 'colab_file_path'].split('/')[-1]}")
    labels_df.loc[idx, 'colab_samples_path'] = audio_slice_file_path
    if IS_RESAMPLE:
        if colab_file_path != labels_df.loc[idx, 'colab_file_path']:
            colab_file_path = labels_df.loc[idx, 'colab_file_path']
        audio = AudioSegment.from_mp3(labels_df.loc[idx, 'colab_file_path'])
        start_pos = labels_df.loc[idx, 'start'] * 1000
        end_pos = labels_df.loc[idx, 'end'] * 1000
        audio_slice = audio[start_pos:end_pos]
        audio_slice.export(audio_slice_file_path, format='mp3')
labels_df = labels_df.reset_index(names='old_index')

  0%|          | 0/882 [00:00<?, ?it/s]

In [6]:
labels_df.head()

,old_index,file_upload,start,end,difference,label,colab_file_path,label_id,colab_samples_path
0,2,1_In_a_restaurant.mp3,5.900917,6.019838,118.921208,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18,/home/alexeysvatov/projects/ProjectLinguist/ju...
1,3,1_In_a_restaurant.mp3,6.230580,6.454639,224.059099,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18,/home/alexeysvatov/projects/ProjectLinguist/ju...
2,4,1_In_a_restaurant.mp3,6.019880,6.230599,210.718572,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15,/home/alexeysvatov/projects/ProjectLinguist/ju...
3,5,1_In_a_restaurant.mp3,6.454629,6.617893,163.263443,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15,/home/alexeysvatov/projects/ProjectLinguist/ju...
4,6,1_In_a_restaurant.mp3,6.617900,6.870468,252.567449,low_rise,/home/alexeysvatov/projects/ProjectLinguist/ju...,10,/home/alexeysvatov/projects/ProjectLinguist/ju...


In [7]:
data = []
hop_length = 128
n_mels = 40
for csp in labels_df.colab_samples_path:
    data.append(
        {
            'colab_samples_path': csp,
            **preprocess_audio(csp, hop_length=hop_length, n_mels=n_mels)
        }
    )

In [8]:
df = pd.DataFrame(data)
df.head()

,colab_samples_path,melspectrum,mfcc,rolloff,zrate
0,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[0.015859658, 0.13360345, 23.94255, 40.439667,...","[-105.55614, 111.83503, -19.678083, -9.315444,...",1290.197754,0.046387
1,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[0.26912397, 46.879185, 70.071, 14.877626, 42....","[-71.38629, 102.2122, -25.945858, 7.254818, 6....",2026.274414,0.061426
2,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[0.16376561, 13.039576, 124.42403, 44.80544, 0...","[-88.1492, 64.30096, -17.069317, 22.782125, 1....",3487.302246,0.047021
3,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[1.1447085, 12.507581, 19.20927, 60.94485, 4.8...","[-103.95176, 59.761208, -0.5934552, 25.225636,...",3779.077148,0.079346
4,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[4.029156, 10.177817, 48.085182, 11.019594, 7....","[-122.036545, 68.16472, 7.255043, 40.853554, 9...",3584.299538,0.063477


In [9]:
mfcc_columns = [f'mfcc_{idx}' for idx in range(df.mfcc[0].size)]
melspectrum_columns = [f'melspectrum_{idx}' for idx in range(df.melspectrum[0].size)]

In [10]:
df[mfcc_columns] = pd.DataFrame(df.mfcc.tolist(), df.index)
df[melspectrum_columns] = pd.DataFrame(df.melspectrum.tolist(), df.index)
df.head()

,colab_samples_path,melspectrum,mfcc,rolloff,zrate,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,...,melspectrum_30,melspectrum_31,melspectrum_32,melspectrum_33,melspectrum_34,melspectrum_35,melspectrum_36,melspectrum_37,melspectrum_38,melspectrum_39
0,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[0.015859658, 0.13360345, 23.94255, 40.439667,...","[-105.55614, 111.83503, -19.678083, -9.315444,...",1290.197754,0.046387,-105.556137,111.835030,-19.678083,-9.315444,-14.854431,...,0.000371,0.000027,0.000404,0.000794,0.001419,0.000512,0.000227,0.000078,0.000007,0.000003
1,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[0.26912397, 46.879185, 70.071, 14.877626, 42....","[-71.38629, 102.2122, -25.945858, 7.254818, 6....",2026.274414,0.061426,-71.386292,102.212196,-25.945858,7.254818,6.239717,...,0.000789,0.000760,0.000855,0.000460,0.000621,0.000698,0.000503,0.000890,0.000269,0.000174
2,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[0.16376561, 13.039576, 124.42403, 44.80544, 0...","[-88.1492, 64.30096, -17.069317, 22.782125, 1....",3487.302246,0.047021,-88.149200,64.300957,-17.069317,22.782125,1.987356,...,0.011107,0.007841,0.008501,0.010087,0.009009,0.006633,0.006096,0.001263,0.000523,0.000133
3,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[1.1447085, 12.507581, 19.20927, 60.94485, 4.8...","[-103.95176, 59.761208, -0.5934552, 25.225636,...",3779.077148,0.079346,-103.951759,59.761208,-0.593455,25.225636,7.580552,...,0.015673,0.007521,0.006390,0.008021,0.011789,0.012451,0.002298,0.001521,0.000926,0.000685
4,/home/alexeysvatov/projects/ProjectLinguist/ju...,"[4.029156, 10.177817, 48.085182, 11.019594, 7....","[-122.036545, 68.16472, 7.255043, 40.853554, 9...",3584.299538,0.063477,-122.036545,68.164719,7.255043,40.853554,9.603881,...,0.018823,0.005997,0.002302,0.001437,0.000822,0.000585,0.001291,0.000896,0.001189,0.000204


In [11]:
labels_df = labels_df.join(
    df.loc[:, ~df.columns.isin(['mfcc', 'melspectrum'])], 
    lsuffix='colab_samples_path',
    rsuffix='colab_samples_path',
)
labels_df.head()

,old_index,file_upload,start,end,difference,label,colab_file_path,label_id,colab_samples_pathcolab_samples_path,colab_samples_pathcolab_samples_path,...,melspectrum_30,melspectrum_31,melspectrum_32,melspectrum_33,melspectrum_34,melspectrum_35,melspectrum_36,melspectrum_37,melspectrum_38,melspectrum_39
0,2,1_In_a_restaurant.mp3,5.900917,6.019838,118.921208,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18,/home/alexeysvatov/projects/ProjectLinguist/ju...,/home/alexeysvatov/projects/ProjectLinguist/ju...,...,0.000371,0.000027,0.000404,0.000794,0.001419,0.000512,0.000227,0.000078,0.000007,0.000003
1,3,1_In_a_restaurant.mp3,6.230580,6.454639,224.059099,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18,/home/alexeysvatov/projects/ProjectLinguist/ju...,/home/alexeysvatov/projects/ProjectLinguist/ju...,...,0.000789,0.000760,0.000855,0.000460,0.000621,0.000698,0.000503,0.000890,0.000269,0.000174
2,4,1_In_a_restaurant.mp3,6.019880,6.230599,210.718572,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15,/home/alexeysvatov/projects/ProjectLinguist/ju...,/home/alexeysvatov/projects/ProjectLinguist/ju...,...,0.011107,0.007841,0.008501,0.010087,0.009009,0.006633,0.006096,0.001263,0.000523,0.000133
3,5,1_In_a_restaurant.mp3,6.454629,6.617893,163.263443,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15,/home/alexeysvatov/projects/ProjectLinguist/ju...,/home/alexeysvatov/projects/ProjectLinguist/ju...,...,0.015673,0.007521,0.006390,0.008021,0.011789,0.012451,0.002298,0.001521,0.000926,0.000685
4,6,1_In_a_restaurant.mp3,6.617900,6.870468,252.567449,low_rise,/home/alexeysvatov/projects/ProjectLinguist/ju...,10,/home/alexeysvatov/projects/ProjectLinguist/ju...,/home/alexeysvatov/projects/ProjectLinguist/ju...,...,0.018823,0.005997,0.002302,0.001437,0.000822,0.000585,0.001291,0.000896,0.001189,0.000204


# learning

In [12]:
X_cols = ['rolloff', 'zrate', *mfcc_columns, *melspectrum_columns]
X = labels_df.loc[:, X_cols]
X.head()

,rolloff,zrate,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,...,melspectrum_30,melspectrum_31,melspectrum_32,melspectrum_33,melspectrum_34,melspectrum_35,melspectrum_36,melspectrum_37,melspectrum_38,melspectrum_39
0,1290.197754,0.046387,-105.556137,111.835030,-19.678083,-9.315444,-14.854431,-3.257582,-7.966661,3.557627,...,0.000371,0.000027,0.000404,0.000794,0.001419,0.000512,0.000227,0.000078,0.000007,0.000003
1,2026.274414,0.061426,-71.386292,102.212196,-25.945858,7.254818,6.239717,-2.439204,7.199624,-11.795731,...,0.000789,0.000760,0.000855,0.000460,0.000621,0.000698,0.000503,0.000890,0.000269,0.000174
2,3487.302246,0.047021,-88.149200,64.300957,-17.069317,22.782125,1.987356,4.919394,-0.475697,-4.415099,...,0.011107,0.007841,0.008501,0.010087,0.009009,0.006633,0.006096,0.001263,0.000523,0.000133
3,3779.077148,0.079346,-103.951759,59.761208,-0.593455,25.225636,7.580552,9.860353,2.934902,-3.946150,...,0.015673,0.007521,0.006390,0.008021,0.011789,0.012451,0.002298,0.001521,0.000926,0.000685
4,3584.299538,0.063477,-122.036545,68.164719,7.255043,40.853554,9.603881,1.962382,6.862002,-0.832480,...,0.018823,0.005997,0.002302,0.001437,0.000822,0.000585,0.001291,0.000896,0.001189,0.000204


In [13]:
y = labels_df.label_id
y.head()

0    18
1    18
2    15
3    15
4    10
Name: label_id, dtype: int8

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
def model_assess(model):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return round(accuracy_score(y_test, preds), 5)

In [16]:
models = {
    'GaussianNB': {
        'model': GaussianNB
    },
    'KNeighborsClassifier': {
        'model': KNeighborsClassifier,
        'params': {
            'n_neighbors': range(5, 100),
            'weights': ('uniform', 'distance'),
            'p': (1, 2)
        }
    },
    'SVC': {
        'model': SVC,
        'params': {
            'decision_function_shape': ('ovo', 'ovr'),
            'kernel': ('linear', 'poly', 'rbf', 'sigmoid', 'precomputed')
        }
    }, 
    'LogisticRegression': {
        'model': LogisticRegression,
        'params': {
            'penalty': ('l1', 'l2', 'elasticnet'),
            'random_state': (42,),
            'solver': ('lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'),
            'multi_class': ('multinomial', 'ovr')
        }
    },
    'RandomForestClassifier': {
        'model': RandomForestClassifier,
        'params': {
            'n_estimators': range(100, 1_001, 100),
            'max_depth': range(10, 51, 10),
            'criterion': ('gini', 'entropy', 'log_loss'),
            'random_state': (42,)
        }
    },
    'MLPClassifier': {
        'model': MLPClassifier,
        'params': {
            'solver': ('adam', 'lbfgs', 'sgd'),
            'max_iter': (2_000,),
            'alpha': tuple(1e-5 * _ for _ in (1, 10, 100, 1_000)),
            'hidden_layer_sizes': tuple(itertools.product(range(10, 101, 5), range(10, 101, 5))),
            'random_state': (42,)
        }
    }
}

In [17]:
stats = []
best_accuracy = 0.0

In [ ]:
for name, raw_model in models.items():
    display(len(stats))
    display(name)
    model = raw_model.get('model')
    params = raw_model.get('params', {})
    for vals in itertools.product(*params.values()):
        current_params = dict(zip(params.keys(), vals))
        try:
            parametrised_model = model(**current_params)
            accuracy = model_assess(parametrised_model)
        except Exception as e:
            accuracy = None
            display(e)
        stats.append(
            {
                'name': name,
                'params': current_params,
                'accuracy': accuracy
            }
        )
        if accuracy is not None and accuracy > best_accuracy:
            best_accuracy = accuracy
            display(f'{current_params = }')
            display(f'{accuracy = }')
    display(best_accuracy)

0

'GaussianNB'

'current_params = {}'

'accuracy = 0.0791'

0.0791

...

'MLPClassifier'

/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Sto

"current_params = {'solver': 'adam', 'alpha': 0.0001, 'hidden_layer_sizes': (110, 20), 'random_state': 42}"

'accuracy = 0.37853'

/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/alexeysvatov/projects/ProjectLinguist/jupyter/.venv/lib64/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Sto

In [ ]:
with open (f'models_hl{hop_length}_nm{n_mels}', 'w') as f:
    jsoned = json.dumps(stats, indent=4, ensure_ascii=False)
    f.write(jsoned)

In [ ]:
# # алгорит Баейса
# nb = GaussianNB()
# model_assess(nb, "Naive Bayes")
# # алгоритм k-ближайших соседей
# knn = KNeighborsClassifier(n_neighbors=30)
# model_assess(knn, "KNN")
# # метод опорных векторов
# svm = SVC(decision_function_shape="ovo")
# model_assess(svm, "Support Vector Machine")
# # логистическая регрессия
# lg = LogisticRegression(random_state=42, solver='newton-cg', multi_class='multinomial')
# model_assess(lg, "Logistic Regression")
# # случайный лес
# rforest = RandomForestClassifier(n_estimators=10_000, max_depth=50, criterion='entropy', random_state=42)
# model_assess(rforest, "Random Forest")
# # многослойный персептрон
# nn = MLPClassifier(solver='adam', alpha=1e-5, hidden_layer_sizes=(5000, 10), random_state=1)
# model_assess(nn, "Neural Nets")